# Task 3 — Notebook 01: Pixel panel (SMAP + CDL)

**NAFSI Task 3** uses **processed weekly SMAP Parquet** (`data/processed/smap/smap_weekly_{year}_wide.parquet`) — **no interim NetCDF**.

Spatial subset: **Task 2 rotation-eligible pixels** (`data/processed/task2/rotation_metrics.parquet`, ~2.08M `iy, ix`) so joins stay tractable and align with rotation analysis.

**Outputs:** `data/processed/task3/task3_pixel_panel.parquet` (`iy`, `ix`, `cdl_2019`) for notebooks 02–03.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

_cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Repo root not found")
sys.path.insert(0, str(REPO_ROOT))

from src.io.smap_weekly_parquet import smap_wide_parquet_path
from src.modeling.task3_smap_anomalies import attach_cdl_2019, load_rotation_eligible_pixels

with open(REPO_ROOT / "configs" / "task3_soil_moisture.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

out_dir = REPO_ROOT / cfg["output"]["processed_dir"]
out_dir.mkdir(parents=True, exist_ok=True)

pixels = load_rotation_eligible_pixels(REPO_ROOT)
panel = attach_cdl_2019(REPO_ROOT, pixels)
out_pq = out_dir / "task3_pixel_panel.parquet"
panel.to_parquet(out_pq, index=False)
print("Wrote", out_pq.relative_to(REPO_ROOT), "rows", len(panel))

# Sanity: one week of 2019 soil moisture on subset (histogram)
y = 2019
wcol = "w019"
path = smap_wide_parquet_path(REPO_ROOT, y)
sm = pd.read_parquet(path, columns=["iy", "ix", wcol]).merge(panel[["iy", "ix"]], on=["iy", "ix"], how="inner")[wcol].astype("float32")
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(sm.dropna().to_numpy(), bins=60, color="#3498db", alpha=0.85)
ax.set_title(f"SMAP {wcol} (2019) on rotation-eligible pixels — distribution")
ax.set_xlabel("sm_surface (m3/m3)")
ax.set_ylabel("count")
fig.tight_layout()
fig_dir = REPO_ROOT / cfg["output"]["figures_dir"]
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "task3__smap_week_histogram_subset.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved", (fig_dir / "task3__smap_week_histogram_subset.png").relative_to(REPO_ROOT))
